---
title: AI-QEM by LTM
description: Mitigate quantum errors with LTM AI-QEM, a Function using GCN-Transformer models with no additional QPU runtime.
---

{/* cspell:ignore França, Hiroyasu, Khatri, Kristan, Nobuyuki, Quek, Ryuji, Sagawa, Sergey, Stilck, Tajima, Takagi, Takahiro, Tsubouchi */}

# AI-QEM by LTM

*See the [API reference](/docs/functions/ltm-ai-qem/api-reference).*

<Admonition type="note">
  IBM Quantum&reg; Functions are a beta feature available only to IBM Quantum Premium Plan, Flex Plan, and On-Prem (through the IBM Quantum Platform API) Plan users. Beta features are subject to change.
</Admonition>

<Admonition type="note" title="Current limitations">
  The current release of **AI-QEM** has the following limitations:

- **Mitigation is limited to the [Heron processor](/docs/ibm-quantum-compute/processor-types#heron) family.** LTM plans to support additional backends in future versions.
- **Error mitigation is applied selectively.** Mitigation is performed only when the absolute expectation value satisfies **$|\langle O \rangle_{noisy}| \geq 0.1$**, so that mitigation is applied only where it is expected to provide meaningful benefit.

These limitations reflect the current scope of AI-QEM. Future releases might relax them as LTM adds support for additional workflows, backends, and features.
</Admonition>

## Overview

Quantum error mitigation (QEM) encompasses algorithmic protocols, such as zero-noise extrapolation (ZNE) [[2]](#references) and probabilistic error cancellation (PEC) [[3]](#references), that are designed to suppress noise-induced bias in expectation values on noisy intermediate-scale quantum (NISQ) hardware, without the large qubit overhead of full fault tolerance. These methods are critical for extracting reliable physical simulations and benchmark results from current processors. Although many QEM methods have been developed in recent years, almost all of them share a fundamental trade-off: they rely on significantly more QPU time to achieve accuracy. Their defining limitation is an exponential sampling overhead, because reconstructing an unbiased signal from noisy circuits greatly increases the statistical variance of the estimator. Consequently, overcoming higher noise levels requires orders of magnitude more measurement shots, which leads to significantly longer QPU runtimes and substantial hardware bottlenecks [[4, 5, 6]](#references).

![Diagram of the AI-QEM workflow. A user-provided Estimator PUB and IBM backend coupling and calibration data feed into the AI-QEM circuit function, which decomposes observables into Pauli strings, measures noisy Pauli expectation values on IBM Quantum hardware, predicts correction factors with an LTM AI model, and applies the corrections to compute mitigated expectation values. A bar chart compares noisy, mitigated, and ideal values for five observables.](/docs/images/guides/ltm-ai-qem/workflow.svg "AI-QEM workflow")

With AI-QEM, you can run quantum circuits on noisy QPUs and obtain improved expectation values compared to other AI-based error mitigation techniques, without any additional QPU runtime overhead. The approach uses a deep neural network with a graph convolutional network (GCN)-Transformer architecture. The network is trained on features that include circuit structure, backend topology, Pauli observables, and qubit calibrations, and it predicts a correction factor, denoted by $\zeta$. Dividing the noisy expectation value by $\zeta$ produces an improved expectation value. Experiments show that AI-QEM produces better results than other AI-based mitigation methods, particularly for circuits with large depth (gate layers) and width (number of qubits), which makes it well suited for utility-scale quantum experiments. The table in the [Benchmarks](#benchmarks) section shows that the GCN-Transformer-based model surpasses random forest (RF)-based models in the 80- to 150-qubit range, with a mean absolute error (MAE) of 0.343. The MAE is the average, over a large number of circuit and observable configurations, of the absolute difference between the ideal expectation value and the expectation value after the model mitigates the noisy hardware result. Furthermore, because the model is circuit-agnostic, you can apply it to a wide range of quantum circuits and applications without circuit-specific customization.

For a full description of the underlying components and a utility-scale demonstration, refer to the AI-QEM paper [[1]](#references).

## Description

Use the LTM AI-QEM function to estimate the expectation values of observables for a given quantum circuit. To use LTM AI-QEM, provide a quantum circuit, a set of observables to measure, and a QPU. The function uses the same QPU time as the IBM Quantum Estimator primitive, and no additional QPU time.

The usual way to compute the expectation value of an observable $O$ is to decompose the observable into a sum of Pauli operators, $O=\sum_i{c_i}{P_i}$, and calculate the expectation value of each Pauli operator, $\langle P_i \rangle$, individually. The expectation value of the observable, $\langle O \rangle$, is then the same linear combination of the Pauli expectation values: ${\langle O \rangle}=\sum_i{c_i}{\langle P_i \rangle}$. If a set of Pauli operators commute with each other, the measurements required to compute their expectation values can be done simultaneously. When you run a circuit with the LTM AI-QEM function, the function uses the Estimator primitive to run the circuit on hardware as usual. It follows the same procedure of decomposing the observable into Pauli operators and computing their expectation values individually. Then the GCN-Transformer, a deep learning architecture, predicts a correction factor $\zeta_i$ for each Pauli operator $P_i$. The function uses this correction factor to correct the noisy expectation value of each Pauli operator: ${\langle P_i \rangle}_{mitigated} = \frac{{\langle P_i \rangle}_{noisy}}{\zeta_i}$. The mitigated expectation value is therefore ${\langle O \rangle}_{mitigated} = \sum_i{c_i}{\langle {P_i} \rangle}_{mitigated} = \sum_i{c_i}{\frac{{\langle P_i \rangle}_{noisy}}{\zeta_i}}$.

The workflow for using the LTM AI-QEM function is as follows:

1. Define your circuits and observables.
2. Run the LTM AI-QEM function by passing a circuit and observable pairs as primitive unified blocs (PUBs).
3. Retrieve the results, which contain the expectation value of the observable with respect to the corresponding quantum circuit in each PUB.

Many test examples empirically show that the LTM AI-QEM function improves the estimation of observables. The function has the following features:

- **Utility-scale experiments:** Test examples show that AI-QEM performs better than other AI-based methods for circuits with large depth and large width. See the [Benchmarks](#benchmarks) section for results.
- **Primitive:** The function supports the Estimator primitive, so you can use it to compute the expectation value of any observable with respect to any quantum circuit.
- **Primitive unified bloc (PUB):** The LTM AI-QEM function supports all types of PUBs (that is, all circuit-observable configurations) that the Estimator primitive supports.

The AI-QEM technique begins with dataset construction, in which random quantum circuits are generated with various parameter settings. For each circuit, both the ideal expectation values (obtained through simulation) and the noisy expectation values (obtained from hardware execution) are computed. These results, along with the corresponding quantum circuits and backend information, are stored in a dataset repository. The dataset is then used to train a machine-learning or deep-learning model that can mitigate noise in quantum computations, and the trained model is saved for future use. During inference, the trained model takes as input a quantum circuit, its noisy expectation value computed on hardware, the relevant observables, and backend information. The model then processes these inputs to produce a noise-mitigated expectation value.

## Benchmarks

The following table shows the performance of the LTM AI-QEM function for different types of circuits, circuit configurations, and observable configurations.

| Circuit | Qubits | Processor | Description | Accuracy | Time (QPU / CPU) |
| --- | --- | --- | --- | --- | --- |
| Mirror circuit (RealAmplitudes) | 30 | Heron r2 | reps = 1, Two-qubit depth = 58, Observable Pauli weight = 1 | 96% | QPU: 14 s<br/>CPU: 4 s |
| Mirror circuit (TwoLocal) | 35 | Heron r2 | reps = 1, Two-qubit depth = 68, Observable Pauli weight = 1 | 81% | QPU: 14 s<br/>CPU: 4 s |
| Mirror circuit (EfficientSU2) | 40 | Heron r2 | reps = 1, Two-qubit depth = 78, Observable Pauli weight = 1 | 95% | QPU: 14 s<br/>CPU: 4 s |
| Clifford circuit | 50 | Heron r2 | Two-qubit depth = 35, Observable Pauli weight = 10 | 95.53% | QPU: 23 s<br/>CPU: 30 s |
| Clifford circuit | 50 | Heron r2 | Two-qubit depth = 40, Observable Pauli weight = 12 | 84% | QPU: 22 s<br/>CPU: 51 s |
| Clifford circuit | 50 | Heron r2 | Two-qubit depth = 30, Observable Pauli weight = 15 | 94.44% | QPU: 22 s<br/>CPU: 54 s |
| Clifford circuit | 100 | Heron r2 | Two-qubit depth = 45, Observable Pauli weight = 5 | 95.06% | QPU: 22 s<br/>CPU: 239 s |
| Clifford circuit | 100 | Heron r2 | Two-qubit depth = 35, Observable Pauli weight = 11 | 94.82% | QPU: 22 s<br/>CPU: 245 s |
| Ising (chaotic angles) | 28 | Heron r2 | Trotter steps = 5, Two-qubit depth = 84, Observable = Average magnetization | 87.29% | QPU: 23 s<br/>CPU: 25 s |
| Ising (chaotic angles) | 36 | Heron r2 | Trotter steps = 6, Two-qubit depth = 228, Observable = Average magnetization | 90.51% | QPU: 23 s<br/>CPU: 31 s |
| Ising (chaotic angles) | 96 | Heron r2 | Trotter steps = 8, Two-qubit depth = 496, Observable = Average magnetization | 90.29% | QPU: 23 s<br/>CPU: 220 s |
| Ising (near-Clifford angles) | 56 | Heron r2 | Trotter steps = 7, Two-qubit depth = 322, Observable = Average magnetization | 89.51% | QPU: 23 s<br/>CPU: 53 s |
| Ising (small angles) | 76 | Heron r2 | Trotter steps = 10, Two-qubit depth = 540, Observable = Average magnetization | 91.36% | QPU: 23 s<br/>CPU: 111 s |

**Note:** CPU time is the post-processing time. The three mirror circuits were submitted together in one run, so the recorded QPU and CPU times are for all three circuits together. Each Clifford circuit run contained nine PUBs with different circuit depths, so the recorded times are for all nine PUBs together. Each Ising circuit run contained 10 PUBs with different numbers of Trotter steps, so the recorded times are for all 10 PUBs together.

For the mirror circuits, the benchmarks use the RealAmplitudes, TwoLocal, and EfficientSU2 ansätze with 30, 35, and 40 qubits, respectively. The observables consist of a single Pauli-$Z$ operator on a randomly selected qubit, with an ideal expectation value of 1.

For the Clifford circuits, the benchmarks use 50- and 100-qubit circuits with observables initialized as one or more randomly positioned Pauli-$Z$ operators. Operator evolution increases the Pauli weight of these observables, which provides more complex observables for benchmarking. The observables are constructed to have an ideal expectation value of 1, which is independently verified with stabilizer simulation.

For the Trotterized 2D tilted-field Ising (TFI) Hamiltonian simulations, the benchmarks use three parameter regimes: small angle $(\theta_X,\theta_Z,\theta_{ZZ})=(0.1,0.1,0.1)$, near-Clifford $(0.1,0.05,1.5708)$, and chaotic $(0.53,0.1,1.0)$. The observable is the average magnetization,

$$
O_M = \frac{1}{M}\sum_i Z_i,
$$

where $M$ is the number of qubits in the simulation. Ideal expectation values are computed with sparse Pauli dynamics (SPD), a Pauli propagation method, and are validated against Qiskit statevector simulations for circuits with fewer than 29 qubits.

For all benchmarks, mitigation accuracy is calculated as

$$
\mathrm{Accuracy}
=
\left(
1-
\frac{
\left|\langle O\rangle_{\mathrm{ideal}}-\langle O\rangle_{\mathrm{mitigated}}\right|
}{
\langle O\rangle_{\mathrm{ideal}}
}
\right)\times100\%,
$$

where $\langle O\rangle_{\mathrm{ideal}}$ and $\langle O\rangle_{\mathrm{mitigated}}$ are the ideal and mitigated expectation values, respectively.

## Get support

The AI-QEM support team is here to help. If you encounter any issues, have questions about using the AI-QEM Function, or need help troubleshooting a job, contact the support team.

Support: quantumfunction.support@ltm.com

To help the support team investigate and resolve your issue as quickly as possible, include the following information in your support request:

- Function job ID (for example, `job.job_id`)
- A detailed description of the issue
- Any relevant error messages, stack traces, or error codes
- Expected behavior and actual behavior (if applicable)

Complete information helps the support team diagnose issues more efficiently and resolve them faster.

The AI-QEM support team also welcomes feedback and feature requests. If you have suggestions for improvements or capabilities that you want to see in future releases, send them to quantumfunction.support@ltm.com.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Run your first workload with the [get started guide](/docs/functions/ltm-ai-qem/get-started).
  - Visit the [API reference](/docs/functions/ltm-ai-qem/api-reference) for this Function.
  - Try the [Trotterized time evolution of the 2D tilted-field Ising model with LTM AI-QEM](/docs/tutorials/trotterized-time-evolution-ltm) tutorial.
  - Review the [functions overview](/docs/guides/functions).
  - Learn about [managed workflows](/docs/managed-workflows).
  - Learn how to choose [problem sizes and compute profiles](/docs/managed-workflows/managed-workflow-compute-profiles).
</Admonition>

## References

[1] Utility Scale Quantum Error Mitigation using AI

[2] [Kristan Temme, Sergey Bravyi, and Jay M. Gambetta, “Error mitigation for short-depth quantum circuits,” Physical Review Letters 119, 180509 (2017).](https://link.aps.org/doi/10.1103/PhysRevLett.119.180509)

[3] [van den Berg, E., Minev, Z.K., Kandala, A. et al. Probabilistic error cancellation with sparse Pauli–Lindblad models on noisy quantum processors. Nat. Phys. 19, 1116–1121 (2023).](https://doi.org/10.1038/s41567-023-02042-2)

[4] [Kento Tsubouchi, Takahiro Sagawa, and Nobuyuki Yoshioka, “Universal cost bound of quantum error mitigation based on quantum estimation theory,” Physical Review Letters 131, 210601 (2023).](https://link.aps.org/doi/10.1103/PhysRevLett.131.210601)

[5] [Quek, Y., Stilck França, D., Khatri, S. et al. Exponentially tighter bounds on limitations of quantum error mitigation. Nat. Phys. 20, 1648–1658 (2024).](https://doi.org/10.1038/s41567-024-02536-7)

[6] [Ryuji Takagi, Hiroyasu Tajima, and Mile Gu, “Universal sampling lower bounds for quantum error mitigation,” Physical Review Letters 131, 210602 (2023).](https://journals.aps.org/prl/pdf/10.1103/PhysRevLett.131.210602)